In [ ]:
# This function reads multiple training log CSV files and plots validation Macro-F1 curves
# according to the user's specification, saving the figure as a PDF and displaying it.

import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict

def plot_flow_ablation_f1_curves(
    log_files: List[str],
    sampling_map: Dict[str, str],
    flow_map: Dict[str, bool],
    output_pdf: str = "flow_ablation_f1_curves.pdf",
    figsize=(8, 5),
    linewidth=2.0,
    fontsize=11,
    alpha=0.95
):
    """
    Plot validation Macro-F1 curves for flow-ablation experiments.

    Parameters
    ----------
    log_files : List[str]
        List of CSV file paths containing training logs.
    sampling_map : Dict[str, str]
        Mapping from log file path -> sampling ratio label (e.g., "3%", "10%", "50%").
    flow_map : Dict[str, bool]
        Mapping from log file path -> whether flow expert is enabled (True/False).
    output_pdf : str
        Path to save the output PDF figure.
    figsize : tuple
        Figure size.
    linewidth : float
        Line width for curves.
    fontsize : int
        Base font size for labels and legend.
    alpha : float
        Line transparency.
    """

    # Color mapping for sampling ratios
    color_map = {
        "3%": "tab:blue",
        "10%": "tab:orange",
        "50%": "tab:green"
    }

    # Line style mapping for flow expert
    linestyle_map = {
        True: "-",    # with flow
        False: "--"   # without flow
    }

    # Read all logs and find maximum epoch
    logs = {}
    max_epoch = 0
    for f in log_files:
        df = pd.read_csv(f)
        logs[f] = df
        max_epoch = max(max_epoch, df["epoch"].max())

    # Plot
    plt.figure(figsize=figsize)

    for f, df in logs.items():
        sampling = sampling_map[f]
        has_flow = flow_map[f]

        label = f"{sampling} | {'With Flow' if has_flow else 'No Flow'}"
        color = color_map[sampling]
        linestyle = linestyle_map[has_flow]

        plt.plot(
            df["epoch"],
            df["val_f1_macro"],
            label=label,
            linestyle=linestyle,
            linewidth=linewidth,
            alpha=alpha,
            color=color
        )

    # Axis and style settings
    plt.xlim(1, max_epoch)
    plt.xlabel("Epoch", fontsize=fontsize)
    plt.ylabel("Validation Macro-F1", fontsize=fontsize)
    plt.xticks(fontsize=fontsize - 1)
    plt.yticks(fontsize=fontsize - 1)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=fontsize - 1, frameon=False)
    plt.tight_layout()

    # Save and display
    plt.savefig(output_pdf, format="pdf")
    plt.show()

    return output_pdf

# Example usage (paths and mappings should be adapted by the user):
# log_files = [
#     "0.03_no_flow.csv", "0.03_flow.csv",
#     "0.10_no_flow.csv", "0.10_flow.csv",
#     "0.50_no_flow.csv", "0.50_flow.csv"
# ]
#
# sampling_map = {
#     "0.03_no_flow.csv": "3%",
#     "0.03_flow.csv": "3%",
#     "0.10_no_flow.csv": "10%",
#     "0.10_flow.csv": "10%",
#     "0.50_no_flow.csv": "50%",
#     "0.50_flow.csv": "50%"
# }
#
# flow_map = {
#     "0.03_no_flow.csv": False,
#     "0.03_flow.csv": True,
#     "0.10_no_flow.csv": False,
#     "0.10_flow.csv": True,
#     "0.50_no_flow.csv": False,
#     "0.50_flow.csv": True
# }
#
# plot_flow_ablation_f1_curves(log_files, sampling_map, flow_map)
